In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [4]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

## MODEO FINAL COM TODOS OS ANOS

In [5]:
#parametro
# k = 1/raiz(xTcovx)
# z = kx


In [6]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]
        retorno_usado = df_usado.copy()

        # COV
        df_cov = retorno_usado.cov()

        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()
    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int+1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])

    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.ky = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.BIGM = pyo.Param(initialize=500)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * model.y[a] * model.k
    # Criando uma auxiliar model.ky que representará model.y[a]*model.k

    # # Com y=0 eu eu preciso que model.ky seja 0
    def ky_zero(model,a):
        return model.ky[a] <= model.BIGM * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=ky_zero)

    # com y=1 ou seja model.ky[a] <= model.y * model.k  (para y=1)
        #  segundo teto
    def res_y_1_teto2(model,a):
        return model.ky[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

        # Limite inf 1, se y é igual a 1, entao model.ky >= model.k   , e dessa forma faz o <= k e >= k , ou seja, ky=k para y=1
    def limite_1(model,a):
        return model.ky[a] >= model.k - model.BIGM*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)

    #  PESOS MIN MAX
    def pesomin(model,a):
        return model.z[a] >= vb_peso_minimo * model.ky[a]
    model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
        return model.z[a] <= vb_peso_maximo * model.ky[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    #  cardinalidade
    def card_min(model):
        return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
        return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
        return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
        return sum(model.z[a]*model.sigma[a,b]*model.z[b] for a in model.ativos for b in model.ativos) <= 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

    def obj_r(model):
        var = sum(sum(model.z[a]*model.retornos_ativos[t,a] for a in model.ativos)for t in model.dias)
        return var
    model.obj = pyo.Objective(rule=obj_r, sense=pyo.maximize)

    # ------------------- solver
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)

    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[ano_int+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass

=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2025
Nao consta model
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2026
## UTILIZANDO DADOS DE RETORNO DE:  2025
2025 -> {'ABEV3': 0.16696607439512473, 'ALOS3': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BBAS3': 0.0, 'BBDC3': 0.0, 'BBDC4': 0.0, 'BBSE3': 0.0, 'BEEF3': 0.0, 'BPAC11': 0.0, 'BRAP4': 0.02000000542457759, 'BRAV3': 0.0, 'BRKM5': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.19707647624073443, 'CSNA3': 0.0, 'CVCB3': 0.0, 'CXSE3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.0, 'ENEV3': 0.0, 'ENGI11': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.0, 'GGBR4': 0.0, 'GOAU4

In [7]:
carteiras_anuais[2026]

{'pesos': {'ABEV3': 0.16696607439512473,
  'ALOS3': 0.0,
  'ANIM3': 0.0,
  'AXIA3': 0.0,
  'AZZA3': 0.0,
  'B3SA3': 0.0,
  'BBAS3': 0.0,
  'BBDC3': 0.0,
  'BBDC4': 0.0,
  'BBSE3': 0.0,
  'BEEF3': 0.0,
  'BPAC11': 0.0,
  'BRAP4': 0.02000000542457759,
  'BRAV3': 0.0,
  'BRKM5': 0.0,
  'CMIG4': 0.0,
  'COGN3': 0.0,
  'CPFE3': 0.0,
  'CPLE3': 0.0,
  'CSAN3': 0.0,
  'CSMG3': 0.19707647624073443,
  'CSNA3': 0.0,
  'CVCB3': 0.0,
  'CXSE3': 0.0,
  'CYRE3': 0.0,
  'DIRR3': 0.0,
  'ECOR3': 0.0,
  'EGIE3': 0.0,
  'EMBJ3': 0.0,
  'ENEV3': 0.0,
  'ENGI11': 0.0,
  'EQTL3': 0.0,
  'EZTC3': 0.0,
  'FLRY3': 0.0,
  'GGBR4': 0.0,
  'GOAU4': 0.0,
  'HYPE3': 0.0,
  'IGTI11': 0.0,
  'IRBR3': 0.0,
  'ISAE4': 0.0,
  'ITSA4': 0.0,
  'ITUB3': 0.0,
  'ITUB4': 0.0,
  'JHSF3': 0.0,
  'KLBN11': 0.0,
  'LREN3': 0.0,
  'MDNE3': 0.0,
  'MGLU3': 0.0,
  'MOTV3': 0.0,
  'MOVI3': 0.0,
  'MRVE3': 0.0,
  'MULT3': 0.0,
  'PETR3': 0.020000007550694327,
  'PETR4': 0.13900073168113009,
  'POMO4': 0.0,
  'PRIO3': 0.1647241553833

In [8]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        # print(v)
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
2016


In [9]:
df_portfolios

,ano,ativo,peso
0,2026,ABEV3,0.1670
1,2026,BRAP4,0.0200
2,2026,CSMG3,0.1971
3,2026,PETR3,0.0200
4,2026,PETR4,0.1390
...,...,...,...
105,2016,HYPE3,0.1742
106,2016,MGLU3,0.0357
107,2016,MRVE3,0.2000
108,2016,SBSP3,0.0692


In [10]:
df_portfolios.to_csv('carteira_linear2_sharpe.csv')


In [16]:
print(df_portfolios[df_portfolios['ano']==2023].peso.sum())
df_portfolios[df_portfolios['ano']==2023]

1.0


,ano,ativo,peso
30,2023,BBSE3,0.2000
31,2023,BRAP4,0.2000
32,2023,CXSE3,0.0200
33,2023,EMBJ3,0.2000
34,2023,KLBN11,0.0200
35,2023,POMO4,0.0738
36,2023,SAPR11,0.0768
37,2023,SUZB3,0.0200
38,2023,VALE3,0.0200
39,2023,WEGE3,0.1694
